# Predictive task training

In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

In [2]:
%load_ext autoreload
%autoreload 2
import os, sys
import pandas as pd
import numpy as np
import pickle as pkl
import torch
import seaborn as sns
import matplotlib.pyplot as plt 
import subprocess

from foundation.clinical import get_batch
from gbmhackathon.data import MosaicDataset
from gbmhackathon.s3_loader import load_s3, write_s3

In [3]:
# BUCKET_MOSAIC = "ABSTRA_DATASET_03bb30aa_16ed_4b89_913e_fe009db2aabd"
# BUCKET_PROJECT = "ABSTRA_PROJECT_STORAGE_BUCKET"

# def fetch_path(env_var_name):
#     return os.path.expandvars(f"${env_var_name}")

# S3_PATH_CLINICAL_EMB = fetch_path(BUCKET_PROJECT) + "embedding_V1/2025-03-30_14-23_clinical_emb_V1.pkl"
# S3_PATH_MODALITIES_PER_SAMPLES = fetch_path(BUCKET_MOSAIC) + "Data availibility per modality per sample.csv"
# S3_PATH_PROJECT = fetch_path(BUCKET_PROJECT)

In [4]:
# clinical_dict = load_s3(S3_PATH_CLINICAL_EMB)

In [5]:
# id2row = clinical_dict['dataset']['id2row']
# X = clinical_dict['dataset']['X']
# Y = clinical_dict['dataset']['Y']
# features = clinical_dict['dataset']['features']
# targets = clinical_dict['dataset']['targets']
# per_mod_contributions = clinical_dict['dataset']['mca_contributions']

In [6]:
# targets

In [7]:
# dict_targets = {pid:Y[id2row[pid],:] for pid in id2row.keys()}
# dict_targets

In [8]:
# freeze = False
# not freeze

In [3]:
%load_ext autoreload
%autoreload 2

from gbmhackathon.training.predictive import *
from gbmhackathon.models.mme import GBMNet
from gbmhackathon.utils.loss_functions import InfoNCELoss, RegularizedInfoNCELoss, SmoothingFunction, RankMe
from gbmhackathon.utils.module_functions import instantiate
from gbmhackathon.s3_loader import load_s3

import os
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# To investigate gradients
from torchviz import make_dot
from sklearn.ensemble import GradientBoostingClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
torch.set_num_threads(12)
torch.get_num_threads()

12

In [5]:
device = "cpu" #"cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

In [6]:
name_emb_dict = {"hne":"embeddings_HnE_OptimusH0.pkl",
#"spatial":"2025-03-23_18-32_spatial_emb_V1.pkl",
"clinical":"2025-03-30_14-23_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl",
"bulk":"2025-05-03_10-15_bulk_emb_V1.pkl",
"scRNA":"2025-05-04_02-35_scRNA_emb_V1.pkl"}
pkl_storage_folder = "embedding_V1"

In [7]:
dataset = PredictiveLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=0.0)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 144
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_predictive, generator=torch.Generator(device=dataset.device))


Using device: cpu
Dataset size: 114


In [9]:
toy_batch = next(iter(dataloader))
#toy_batch

In [10]:
type(toy_batch)

tuple

In [11]:
toy_batch[2]['hne'].size()

torch.Size([114, 1536])

In [12]:
raw_emb = toy_batch[2]
inpute_size_dict = {}
for mod in raw_emb.keys():
    print(mod, raw_emb[mod].size())
    inpute_size_dict[mod] = raw_emb[mod].size(1)

hne torch.Size([114, 1536])
clinical torch.Size([114, 12])
wes torch.Size([114, 1790])
bulk torch.Size([114, 3072])
scRNA torch.Size([114, 3072])


In [13]:
raw_emb = toy_batch[2]

# 1) collecter les tenseurs et construire input_size_dict
tensors_to_concat = []
input_size_dict = {}

for mod, emb in raw_emb.items():
    if isinstance(emb, torch.Tensor):
        # on suppose que emb a la forme [batch_size, feature_size]
        input_size_dict[mod] = emb.size(1)
        tensors_to_concat.append(emb)
    else:
        # si tu veux voir ce qu'on skip :
        print(f"Skip {mod}: not a Tensor but {type(emb)}")

# 2) concaténation
# attention tous les emb doivent avoir le même batch_size (dim 0)
concat_emb = torch.cat(tensors_to_concat, dim=1)

print("Sizes by module:", input_size_dict)
print("Concatenated tensor size:", concat_emb.size())

Sizes by module: {'hne': 1536, 'clinical': 12, 'wes': 1790, 'bulk': 3072, 'scRNA': 3072}
Concatenated tensor size: torch.Size([114, 9482])


In [14]:
X=concat_emb

In [15]:
Y=toy_batch[-2]

In [16]:
y_reg=Y[:,:3]
y_cat=Y[:,3:]

In [17]:
print(y_reg.size(),y_cat.size())

torch.Size([114, 3])

In [19]:
print(X.size(),Y.size())

torch.Size([114, 9482]) torch.Size([114, 5])


In [23]:
from sklearn.multioutput import MultiOutputRegressor, MultiOutputClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, f1_score
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier


# X : features, y_reg : array shape (n_samples,3), y_cat : array shape (n_samples,2)
loo = LeaveOneOut()
rmse_sums = np.zeros(3)
f1_sums = np.zeros(2)
n = X.shape[0]

for i, (train_idx, test_idx) in enumerate(loo.split(X), 1):
    
    
#for train_idx, test_idx in loo.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_reg_train, y_reg_test = y_reg[train_idx], y_reg[test_idx]
    y_cat_train, y_cat_test = y_cat[train_idx], y_cat[test_idx]

    # Meta-estimateurs multitâche
    #reg_mt = MultiOutputRegressor(GradientBoostingRegressor())       # régression 3 sorties :contentReference[oaicite:3]{index=3}
    #clf_mt = MultiOutputClassifier(GradientBoostingClassifier())      # classification 2 sorties :contentReference[oaicite:4]{index=4}


    reg_mt = MultiOutputRegressor(
        HistGradientBoostingRegressor(max_iter=100, early_stopping=True),
        n_jobs=-1)
    clf_mt = MultiOutputClassifier(
        HistGradientBoostingClassifier(max_iter=100, early_stopping=True),
        n_jobs=-1)

    
    # Entraînement
    reg_mt.fit(X_train, y_reg_train)
    clf_mt.fit(X_train, y_cat_train)

    # Prédiction
    y_reg_pred = reg_mt.predict(X_test)
    y_cat_pred = clf_mt.predict(X_test)

    # Accumulation des métriques
    for k in range(3):
        rmse_sums[k] += np.sqrt(mean_squared_error(y_reg_test[:, k], y_reg_pred[:, k]))
    for j in range(2):
        f1_sums[j] += f1_score(y_cat_test[:, j], y_cat_pred[:, j], average='macro')
        
    print(f"Iteration {i}/{n}", end="\r")
    
# Moyennes LOO
rmse_means = rmse_sums / n
f1_means = f1_sums / n

print("Multitâche LOO → RMSE:", rmse_means, "— F1:", f1_means)


Multitâche LOO → RMSE: [0.75757657 0.70900914 0.75248359] — F1: [0.81578947 0.85087719]
